In [11]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from preprocessing import (RANDOM_STATE, columns_type, fold_matrices, held_out_ap,
                           load_data, paired_delta, target, temporal_splits)
import warnings
from collections import namedtuple
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



train_clientes = load_data()
numeric, categorical = columns_type(train_clientes)
frame = train_clientes[numeric + categorical + [target]]
time_splits = temporal_splits(frame)

# Provisional recipe: rules only, until 1.7.2 and 1.8 are closed
recipe = {
    "numeric": numeric,
    "categorical": categorical,
    "fill": [(numeric, SimpleImputer(strategy="median")),
             (categorical, SimpleImputer(strategy="most_frequent"))],
    "log1p": [],  # the outlier treatment chosen in 1.7.2 goes here
}


print(len(numeric), "numeric |", len(categorical), "categorical |", len(time_splits), "splits")

37 numeric | 12 categorical | 7 splits


Before moving on to the feature selection techniques, we need to set a **reference** for the results given by our metric (**AP** -> average precision), so that we can then compare them with the results obtained when the techniques are applied.

The function `held_out_ap` receives four parameters:

1. `frame` (the data)
2. `time_splits` (the 7 temporal splits: each one trains on the past and validates on the period right after it)
3. `recipe` (The pre-processing methods defined in the previus notebooks)
4. `select` (receives the feature selection technique to apply inside each split)

> **Note:** the reference level leaves it empty, meaning no feature selection technique was used.

After getting these, for each split it will prepare the training and validation rows with the recipe, finally trains a logistic regression on the training rows and scores it on the validation rows.

In [12]:
reference = held_out_ap(frame, time_splits, recipe)
print(reference[["n", "AP", "F1"]])
print(f"mean: AP {reference['AP'].mean():.4f} | F1 {reference['F1'].mean():.4f}")

     n        AP        F1
0  562  0.824279  0.701987
1  585  0.838774  0.699213
2  624  0.853865  0.745583
3  650  0.817163  0.722045
4  694  0.801795  0.687386
5  753  0.858422  0.708464
6  770  0.854445  0.758824
mean: AP 0.8355 | F1 0.7176


---

## Feature Selection

Feature Selection is the process of selecting the most relevant features from a dataset to improve model performance. To simplify the model, this section aims to reduce the number of features, reduce computational complexity and potentially improve its generalization ability.

### Filter Methods 

Filter methods are one of the feature selection techniques that evaluate the relevance of features based on statistical measures without involving a machine learning model.

Selects the top-ranked features and removes irrelevant or redundant ones before model training.

#### Variance Thresholding

Variance Thresholding removes features with very low variance, as they contribute little useful information to the learning process. Features with almost constant values are typically discarded.

In [13]:
def report(name, run):
    """File one technique's rows on the board, then say what it scored and kept.

    The verdict comes first, in words, against the all-features reference. The
    kept set follows as a FREQUENCY over the splits, and only the columns kept
    in most of them are named.
    """
    board[name] = run
    kept = Counter(column for columns in run["kept"] for column in columns)
    row = {"columns": run["n"].mean(), "F1": run["F1"].mean()}
    if ("evaluation" in run
            and run["evaluation"].eq("model-selection validation").all()):
        row["verdict"] = "no unbiased delta"
        line = (f"validation F1 {row['F1']:.4f} with {row['columns']:.1f} columns;"
                " these splits chose this width, so no gap to all features is claimed")
    elif name == "all features":
        line = (f"F1 {row['F1']:.4f} with {row['columns']:.1f} columns: the reference"
                " every other row is compared with")
    else:
        row.update(paired_delta(run["F1"], reference["F1"]))
        line = (f"F1 {row['F1']:.4f} with {row['columns']:.1f} columns, against"
                f" {reference['F1'].mean():.4f} with all features: dF1 {row['dF1']:+.4f}"
                f" +- {row['SEM']:.4f}, better on {row['wins']} splits"
                f" -> {row['verdict'].upper()}")
    results[name] = row
    usual = [f"{c} {n}/{len(splits)}"
             for c, n in kept.most_common() if n > len(splits) / 2]
    rare = sum(n <= len(splits) / 2 for n in kept.values())
    print(name)
    print(textwrap.fill(line, width=84, initial_indent="    ",
                        subsequent_indent="    "))
    print(textwrap.fill("kept in most splits: " + ", ".join(usual), width=84,
                        initial_indent="    ", subsequent_indent="      "))
    if rare:
        print(f"    kept in half the splits or fewer: {rare} more"
              f" column{'' if rare == 1 else 's'}")

In [14]:
# Split 0's training matrix, prepared by the recipe
train_rows, validation_rows = time_splits[0]
A_train, _, names = fold_matrices(frame.iloc[train_rows], frame.iloc[validation_rows], recipe)

variances = pd.Series(A_train.var(axis=0), index=names)
measured = variances.index.isin(numeric)
print(f"{len(variances)} columns, {int((variances == 0).sum())} constant")
print(f"dummies : variance {variances[~measured].min():.5f} to {variances[~measured].max():.4f}")
print(f"measured: variance {variances[measured].min():.4f} to {variances[measured].max():.4f}")

562 columns, 0 constant
dummies : variance 0.00045 to 0.2297
measured: variance 1.0000 to 1.0000


In [15]:

variance = held_out_ap(frame, time_splits, recipe, select=lambda: VarianceThreshold(0.0))
print(variance[["n", "AP", "F1"]])
print("AP:", paired_delta(variance["AP"], reference["AP"]))
print("F1:", paired_delta(variance["F1"], reference["F1"]))

     n        AP        F1
0  562  0.824279  0.701987
1  585  0.838774  0.699213
2  624  0.853865  0.745583
3  650  0.817163  0.722045
4  694  0.801795  0.687386
5  753  0.858422  0.708464
6  770  0.854445  0.758824
AP: {'delta': np.float64(0.0), 'SEM': np.float64(0.0), 'wins': '0/7', 'verdict': 'identical on every split'}
F1: {'delta': np.float64(0.0), 'SEM': np.float64(0.0), 'wins': '0/7', 'verdict': 'identical on every split'}


**Conclusions:** The filter removed zero columns because a threshold of 0.0 only drops features with zero variance. Since no columns were constant, the dataset remained unchanged.

### Wrapper Methods 

### Embedded Methods 

### Feature Extraction

## Dimensionality Reduction 